# Precise pupil and movement annotation
Run each block separately. Finish the interaction in one block before starting the next. The source videos remain read-only.

In [ ]:
# Block P1 - environment and read-only alignment context
%matplotlib widget
from pathlib import Path
from types import SimpleNamespace
import pandas as pd
import matplotlib.pyplot as plt
from attention_alignment import load_config
from attention_alignment.behavior import (
    NotebookLandmarkSelector,
    NotebookPolygonSelector,
    load_roi_config,
    pupil_audit_frame_indices,
    preview_pupil_detection,
    preview_saved_pupil_detections,
    preview_pupil_thresholds,
    representative_pupil_review_frame_indices,
    region_from_mapping,
    save_camera_annotation,
    save_pupil_manual_anchor,
    summarize_pupil_tracking,
)
from attention_alignment.pipeline import build_alignment, export_session

CONFIG_PATH = Path('configs/sessions.local.yaml')
CONFIG = load_config(CONFIG_PATH)
RESULTS = build_alignment(CONFIG, dry_run=True)
print(f'Loaded {len(RESULTS)} sessions')

In [ ]:
# Block P2 - choose one session/camera and its annotation frame
SESSION_ID = 'replace_with_session_id'
CAMERA = '01'
FRAME_INDEX = 100  # Reviewed reference frame for ROI, pupil seed, and landmarks.
PUPIL_THRESHOLD = None  # None adapts to illumination on every processed frame.
roi_path = CONFIG.project_root / 'configs' / 'rois.local.yaml'
saved_camera_annotation = (
    load_roi_config(roi_path).get(SESSION_ID, {}).get(CAMERA)
)
REUSE_SAVED_ANNOTATION = saved_camera_annotation is not None

result = RESULTS[SESSION_ID]
video_info = result.manifest['videos'][CAMERA]
assert video_info['status'] == 'ok', video_info
video_path = CONFIG.video_path(SESSION_ID, CAMERA)
fps = float(video_info['average_rate_hz'])
frame_count = int(video_info['frame_count_header'])
print(video_path)
print({'fps': fps, 'frame_count': frame_count, 'reference_frame': FRAME_INDEX})
print({'reuse_saved_annotation': REUSE_SAVED_ANNOTATION, 'roi_path': roi_path})

In [ ]:
# Block P3 - eye-aperture polygon (exclude fur and background)
if REUSE_SAVED_ANNOTATION:
    eye_selector = SimpleNamespace(
        roi=region_from_mapping(saved_camera_annotation['eye'])
    )
    print('Loaded saved eye aperture:', eye_selector.roi)
else:
    eye_selector = NotebookPolygonSelector(
        video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} eye aperture'
    )
    print('Click around the visible eye aperture, then click the first point to close.')

In [ ]:
# Block P4 - manually reviewed pupil boundary on the reference frame
if REUSE_SAVED_ANNOTATION and saved_camera_annotation.get('pupil_seed'):
    pupil_seed_selector = SimpleNamespace(
        roi=region_from_mapping(saved_camera_annotation['pupil_seed'])
    )
    print('Loaded saved pupil seed:', pupil_seed_selector.roi)
else:
    pupil_seed_selector = NotebookPolygonSelector(
        video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} pupil boundary seed'
    )
    print('Use 8-16 ordered boundary points and close at the first point.')

In [ ]:
# Block P5 - eye-adjacent stabilization landmarks
LANDMARK_NAMES = (
    'eye_corner_temporal',
    'eye_corner_nasal',
    'fur_anchor_upper',
    'fur_anchor_lower',
)
if REUSE_SAVED_ANNOTATION and saved_camera_annotation.get('eye_landmarks'):
    landmark_selector = SimpleNamespace(
        landmarks=saved_camera_annotation['eye_landmarks']
    )
    print('Loaded saved stabilization landmarks:', landmark_selector.landmarks)
else:
    landmark_selector = NotebookLandmarkSelector(
        video_path,
        FRAME_INDEX,
        names=LANDMARK_NAMES,
        title=f'{SESSION_ID} {CAMERA} stabilization landmarks',
    )
    print('Click stable points in the listed order; avoid pupil glints.')

In [ ]:
# Block P6 - movement polygon
if REUSE_SAVED_ANNOTATION:
    movement_selector = SimpleNamespace(
        roi=region_from_mapping(saved_camera_annotation['movement'])
    )
    print('Loaded saved movement region:', movement_selector.roi)
else:
    movement_selector = NotebookPolygonSelector(
        video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} movement region'
    )
    print('Select textured movement pixels; exclude glare and fixed borders.')

In [ ]:
# Block P7 - compare thresholds; adaptive per-frame mode is recommended
FIXED_THRESHOLD_CANDIDATE = 65  # Preserved from the manual reference-frame review.
PUPIL_THRESHOLD = None  # None adapts to illumination on every frame.
preview_pupil_thresholds(
    video_path,
    eye_selector.roi,
    FRAME_INDEX,
    pupil_seed=pupil_seed_selector.roi,
)
# Use a numeric PUPIL_THRESHOLD only if it remains accurate across the whole P8 audit.
# None evaluates several local-intensity percentiles per frame and uses the temporal prior.

In [ ]:
# Block P8 - audit one frame every two minutes through the complete video
AUDIT_INTERVAL_MINUTES = 2
AUDIT_FRAMES = pupil_audit_frame_indices(
    frame_count,
    fps,
    start_frame=FRAME_INDEX,
    interval_s=AUDIT_INTERVAL_MINUTES * 60,
)
print(f'Previewing {len(AUDIT_FRAMES)} frames from {AUDIT_FRAMES[0]} to {AUDIT_FRAMES[-1]}')
preview_pupil_detection(
    video_path,
    eye_selector.roi,
    AUDIT_FRAMES,
    threshold=PUPIL_THRESHOLD,
    pupil_seed=pupil_seed_selector.roi,
    reference_frame_index=FRAME_INDEX,
    temporal_context_frames=int(round(fps)),
)
# Green=convex measurement boundary; red dashed=secondary ellipse; yellow dotted=eye aperture.
# A large hull-correction percentage means the raw contour was strongly concave and must be reviewed.

In [ ]:
# Block P9 - explicitly persist this reviewed camera annotation
roi_path = CONFIG.project_root / 'configs' / 'rois.local.yaml'
saved_annotation = save_camera_annotation(
    roi_path,
    SESSION_ID,
    CAMERA,
    eye_selector.roi,
    movement_selector.roi,
    pupil_seed=pupil_seed_selector.roi,
    eye_landmarks=landmark_selector.landmarks,
    reference_frame_index=FRAME_INDEX,
    pupil_threshold=PUPIL_THRESHOLD,
)
display(saved_annotation)
display(load_roi_config(roi_path)[SESSION_ID][CAMERA])

In [ ]:
# Block P10 - full-video extraction is an explicit opt-in
RUN_BEHAVIOR = True
if RUN_BEHAVIOR:
    output_dir = export_session(
        CONFIG, result, include_video_indexes=False, include_behavior=True
    )
    behavior_path = output_dir / f'behavior_{CAMERA}.parquet'
    behavior = pd.read_parquet(behavior_path)
    print(behavior_path)
    display(summarize_pupil_tracking(behavior))
    behavior.plot(
        x='t_session_s',
        y=['pupil_area_normalized', 'movement_abs_difference'],
        subplots=True,
        figsize=(14, 6),
    )
else:
    print('Extraction skipped. Set RUN_BEHAVIOR=True only after reviewing Block P8 and P9.')

In [ ]:
# Block P11 - inspect the exact contours saved in behavior parquet
if RUN_BEHAVIOR:
    INVALID_PREVIEW_LIMIT = 12
    MANUAL_REVIEW_FRAMES = [94734, 129877, 141948, 144100]  # Edit freely.
    representative_frames = representative_pupil_review_frame_indices(
        behavior, max_frames=INVALID_PREVIEW_LIMIT
    )
    review_frames = sorted(set(representative_frames + MANUAL_REVIEW_FRAMES))
    review_frames = [index for index in review_frames if 0 <= index < frame_count]
    print(f'Exact saved contours to review: {review_frames}')
    if review_frames:
        preview_saved_pupil_detections(
            video_path, behavior, review_frames, eye_roi=eye_selector.roi
        )
        print('Green=analysis-observed; amber=review/excluded; red=rejected/invalid.')
    else:
        print('No unresolved or automatically flagged pupil frames remain.')

In [ ]:
# Block P12 - optional manual anchor for a difficult frame
# [196, 21459, 40249, 45222, 68953, 89204, 94734, 105502, 116995, 121725, 129488, 129877, 139112, 
# 141948, 144100, 144280]
MANUAL_ANCHOR_FRAME = 113711  # Replace with one frame identified in P11.
MAX_MANUAL_ANCHORS = int(CONFIG.behavior['max_manual_pupil_anchors'])
manual_anchor_selector = NotebookPolygonSelector(
    video_path,
    MANUAL_ANCHOR_FRAME,
    f'{SESSION_ID} {CAMERA} manual pupil anchor frame {MANUAL_ANCHOR_FRAME}',
)
print(
    f'Click 8-16 ordered pupil-boundary points and close at the first point. '
    f'Configured maximum: {MAX_MANUAL_ANCHORS} anchors.'
)

In [ ]:
# Block P13 - save the reviewed anchor; rerun P10 afterward
saved_manual_anchors = save_pupil_manual_anchor(
    roi_path,
    SESSION_ID,
    CAMERA,
    MANUAL_ANCHOR_FRAME,
    manual_anchor_selector.roi,
    max_anchors=MAX_MANUAL_ANCHORS,
)
print({
    'saved_anchor_count': len(saved_manual_anchors),
    'frames': sorted(map(int, saved_manual_anchors)),
    'next_step': 'Rerun P10, then P11.',
})

## P14 - Required order after saving manual anchors

1. Finish all intended corrections with **P12 -> P13** first. The limit comes from `behavior.max_manual_pupil_anchors` in `configs/sessions.yaml`; P13 writes anchors to `configs/rois.yaml`.
2. Run **P10 once** and wait for the new `behavior_<camera>.parquet`, QC summary, and trace plot. P10 reloads the saved anchors from `rois.yaml`; do not interrupt this long extraction.
3. Run **P11** to inspect the exact contours stored in the new behavior parquet. Green is a strict analysis observation, amber is flagged for review/excluded, and red is rejected/invalid. Check every manual anchor and representative flagged frames.
4. If any contour is still anatomically wrong, add the remaining corrections with **P12 -> P13**, then repeat **P10 -> P11**. Do not continue to Notebook 03 until this review passes.
5. Close figures with `plt.close('all')`, then shut down the Notebook 02 kernel to release video-processing memory. Open Notebook 03 with the `attention` kernel.
6. In Notebook 03, run **A1 -> A2 -> A3** first. Continue only when `ready_for_train_and_test_analysis=True` and both train/test phase-valid fractions meet the configured threshold. Otherwise return here for QC.
7. After the A3 gate passes, run **A4 onward in order**. Any Notebook 03 result created before the latest P10 export is stale and must be regenerated.